# 🤖 Giai đoạn 3: Huấn luyện & So sánh các mô hình Machine Learning
Thử nghiệm Naive Bayes, Logistic Regression, SVM, Random Forest & Stacking Classifier


In [ ]:
import sys, os
sys.path.append("..")
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from src.features import FeatureExtractor
from src.models import SentimentModelTrainer

# 1. Đọc dữ liệu đã gán nhãn & làm sạch
df = pd.read_excel("../data/processed/reviews_cleaned.xlsx")

# 2. Vector hóa TF-IDF
fe = FeatureExtractor(method="tfidf", max_features=5000, ngram_range=(1, 2))
X = fe.fit_transform(df["processed_text"])
y = df["sentiment"] if "sentiment" in df.columns else df["Rating"]

# 3. Chia tập train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")


## 3.1 Huấn luyện và so sánh mô hình


In [ ]:
trainer = SentimentModelTrainer()
results_df = trainer.train_and_evaluate_all(X_train, y_train, X_test, y_test)
print(results_df)


## 3.2 Huấn luyện mô hình kết hợp Stacking Ensemble


In [ ]:
stacking_model = trainer.get_stacking_model()
stacking_model.fit(X_train, y_train)
y_pred_stack = stacking_model.predict(X_test)

# Lưu mô hình tốt nhất
trainer.trained_models["Stacking"] = stacking_model
trainer.save_model("Stacking", "../models/best_sentiment_model.joblib")
fe.save("../models/tfidf_vectorizer.joblib")
